# Multi-Color Moran Process -- Toy Validation

Three checks:
1. **Single run** -- visualize how lineages compete over time.
2. **Winner distribution** -- on a complete graph every node should win with equal probability (1/N). This is the key correctness check for neutral drift.
3. **Graph topology comparison** -- complete vs cycle vs star to see if topology shifts the winner distribution.
4. **Winner map** -- graph drawn with each node colored by how often it won.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
from collections import Counter

from moran_process import MultiColorMoranProcess
from moran_process.core.population_graph import PopulationGraph

In [ ]:
def draw_winner_map(pop_graph, winners=None, n_trials=500, ax=None, title=None, layout_seed=42):
    """
    Draw pop_graph with each node colored by its fraction of wins.

    Pass a pre-computed `winners` list to skip re-running simulations,
    otherwise n_trials simulations are run internally.

    Colormap is RdYlGn centered at 1/N (the expected neutral-drift rate):
      yellow = expected, green = above expected, red = below expected.

    Returns win_fractions array (length n_nodes).
    """
    N = pop_graph.number_of_nodes()
    expected = 1.0 / N

    if winners is None:
        winners = []
        for _ in range(n_trials):
            sim = MultiColorMoranProcess(pop_graph)
            sim.initialize_unique_colors()
            winners.append(sim.run()['winner'])

    counts = Counter(winners)
    win_fractions = np.array([counts.get(i, 0) / len(winners) for i in range(N)])

    max_dev = max(abs(win_fractions - expected).max(), expected * 0.5)
    norm = mcolors.TwoSlopeNorm(vmin=expected - max_dev, vcenter=expected, vmax=expected + max_dev)
    node_colors = [plt.cm.RdYlGn(norm(win_fractions[i])) for i in range(N)]

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7, 6))

    pos = nx.spring_layout(pop_graph.graph, seed=layout_seed)
    nx.draw_networkx(
        pop_graph.graph, pos=pos, ax=ax,
        node_color=node_colors,
        node_size=500,
        font_size=8,
        with_labels=N <= 30,
        edge_color='#aaaaaa',
    )

    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Win fraction', shrink=0.8)

    ax.set_title(title or f'{pop_graph.name}  ({len(winners)} trials, expected={expected:.3f})')
    ax.axis('off')

    if own_fig:
        plt.tight_layout()
        plt.show()

    return win_fractions

## 1. Single run -- lineage competition over time

In [ ]:
N = 12
g = PopulationGraph.complete_graph(N)

sim = MultiColorMoranProcess(g)
sim.initialize_unique_colors()
result = sim.run(track_history=True)

print(f"Winner: node {result['winner']}  |  Steps: {result['steps']}  |  Duration: {result['duration']:.4f}s")

In [ ]:
history = result['history']   # shape: (steps, n_nodes)
T, _ = history.shape

# For each step, count how many nodes carry each color
lineage_counts = np.zeros((T, N), dtype=int)
for t in range(T):
    unique, counts = np.unique(history[t], return_counts=True)
    lineage_counts[t, unique] = counts

fig, ax = plt.subplots(figsize=(13, 5))
cmap = plt.cm.tab20(np.linspace(0, 1, N))
for c in range(N):
    alive = lineage_counts[:, c] > 0
    label = f'node {c}' + (' (winner)' if c == result['winner'] else '')
    lw = 2.5 if c == result['winner'] else 1.0
    ax.plot(lineage_counts[:, c], color=cmap[c], label=label, linewidth=lw)

ax.set_xlabel('Step')
ax.set_ylabel('Lineage size (# nodes)')
ax.set_title(f'Multi-color Moran -- complete graph N={N}, winner=node {result["winner"]}')
ax.legend(loc='upper right', ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

## 2. Winner distribution -- complete graph (expect uniform)

On a complete graph with neutral drift every node is equally likely to win, so each node should appear as winner ~`n_trials / N` times.

In [ ]:
N = 15
N_TRIALS = 600
g = PopulationGraph.complete_graph(N)

winners = []
for _ in range(N_TRIALS):
    sim = MultiColorMoranProcess(g)
    sim.initialize_unique_colors()
    winners.append(sim.run()['winner'])

winner_counts = Counter(winners)
expected = N_TRIALS / N

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(N), [winner_counts.get(i, 0) for i in range(N)], color='steelblue', alpha=0.75)
ax.axhline(expected, color='crimson', linestyle='--', linewidth=1.5, label=f'Expected = {expected:.0f}')
ax.set_xlabel('Node index')
ax.set_ylabel('Times won')
ax.set_title(f'Winner distribution -- complete graph N={N}, {N_TRIALS} trials')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Min wins: {min(winner_counts.values())}  Max wins: {max(winner_counts.values())}  Expected: {expected:.1f}")

## 3. Topology comparison -- does graph structure shift the winner?

For neutral drift the winner distribution should remain uniform regardless of topology (the Moran process on any connected graph converges to uniform fixation probabilities under neutrality). Any deviation here would indicate a bug.

In [ ]:
N = 8
N_TRIALS = 1000

graphs = {
    'complete': PopulationGraph.complete_graph(N),
    'cycle':    PopulationGraph.cycle_graph(N),
    'star':     PopulationGraph.star_graph(N),
    'random':   PopulationGraph.random_connected_graph(N, N-1)
}

# Collect winners for each graph
all_winners = {}
for name, g in graphs.items():
    winners = []
    for _ in range(N_TRIALS):
        sim = MultiColorMoranProcess(g)
        sim.initialize_unique_colors()
        winners.append(sim.run()['winner'])
    all_winners[name] = winners

# Draw winner maps using the results we just computed
# Change to 2x2 subplots and optionally adjust figsize for a square layout
fig, axes = plt.subplots(2, 2, figsize=(10, 10)) 

# Flatten the 2D axes array (2x2) into a 1D array of 4 items for the loop
axes_flat = axes.flatten()

for ax, (name, g) in zip(axes_flat, graphs.items()):
    draw_winner_map(g, winners=all_winners[name], ax=ax)

plt.suptitle(f'Winner maps by topology (N={N}, {N_TRIALS} trials each)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Winner map -- graph drawn with nodes colored by win frequency

Nodes are colored on a  scale centered at the expected win rate :
- **Yellow** = wins exactly as expected (neutral)
- **Green** = wins more than expected
- **Red** = wins less than expected

For neutral drift all nodes should be yellow regardless of topology.

In [ ]:
# Side-by-side: complete vs cycle
N = 8
N_TRIALS = 600

graphs = {
    'complete': PopulationGraph.complete_graph(N),
    'cycle':    PopulationGraph.cycle_graph(N),
    'star':     PopulationGraph.star_graph(N),
    'random':   PopulationGraph.random_connected_graph(N, N-1)
}

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for ax, (name, g) in zip(axes, graphs.items()):
    draw_winner_map(g, n_trials=N_TRIALS, ax=ax)

plt.suptitle(f'Winner maps (N={N}, {N_TRIALS} trials each)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Respiratory graphs -- do structurally asymmetric nodes show positional bias?
N_TRIALS = 800

resp_graphs = [
    PopulationGraph.mammalian_lung_graph(),
    PopulationGraph.avian_graph(),
]

fig, axes = plt.subplots(1, len(resp_graphs), figsize=(15, 6))
for ax, g in zip(axes, resp_graphs):
    wf = draw_winner_map(g, n_trials=N_TRIALS, ax=ax)
    print(f"{g.name}: min={wf.min():.3f}  max={wf.max():.3f}  expected={1/g.number_of_nodes():.3f}")

plt.tight_layout()
plt.show()